# CENSD Robustness Experiments (QTDB + NSTDB + SimEMG)

이 노트북은 리뷰 대응용으로 아래 3가지를 **정량 지표로 강조**하기 위한 실험 셋업을 제공합니다.

- **SNR mismatch robustness**: 학습 분포와 테스트 분포가 달라도 성능 하락(Δ)이 작음
- **Tail performance**: 최악 조건(저 SNR, artifact-heavy)에서 덜 무너짐 (worst-k%, quantile)
- **Unseen composition**: 학습에서 제외한 조합/강도 영역에서도 성능 유지

핵심은 두 층으로 나눕니다.

1) **Controlled shift (QTDB + NSTDB)**: clean은 고정, noise 분포만 조작 → 원인/기여를 깔끔히 분해(ablation 가능)
2) **External shift (SimEMG)**: reference(near-clean) 신호가 있는 외부 데이터셋 → 실제 도메인 시프트에서 일반화 확인

In [1]:
import os
import pickle
import numpy as np
from dataclasses import dataclass

DATA_DIR = 'data'
SAMPLES = 512
DEFAULT_SEED = 1234
SNR_LEVELS = [-6, -3, 0, 3, 6, 12, 18]


In [2]:
def load_qtdb_beats(samples=SAMPLES):
    with open(os.path.join(DATA_DIR, 'QTDatabase.pkl'), 'rb') as f:
        qtdb = pickle.load(f)
    beats = []
    for rec in qtdb.keys():
        for b in qtdb[rec]:
            b = np.asarray(b)
            b_np = np.zeros(samples, dtype=np.float32)
            init_padding = 16
            if b.shape[0] > (samples - init_padding):
                continue
            b_np[init_padding:b.shape[0] + init_padding] = b - (b[0] + b[-1]) / 2
            beats.append(b_np)
    beats = np.asarray(beats, dtype=np.float32)
    return beats

clean_beats = load_qtdb_beats()
print('[INFO] clean_beats:', clean_beats.shape, clean_beats.dtype)

[INFO] clean_beats: (85318, 512) float32


In [3]:
def load_noise_component(component: str, snr_db: int):
    # Files created by qtdb.ipynb: shape (2, 650000, 1)
    path = os.path.join(DATA_DIR, f'{component}_Noise_SNR_{snr_db}.pkl')
    with open(path, 'rb') as f:
        noise = pickle.load(f)
    noise = np.asarray(noise)
    if noise.ndim == 3:
        noise = noise.squeeze(-1)
    # now (2, 650000)
    return noise

def load_existing_stream(path):
    with open(os.path.join(DATA_DIR, path), 'rb') as f:
        x = pickle.load(f)
    return np.asarray(x)

# Quick presence check
for comp in ['BW','EM','MA']:
    _ = load_noise_component(comp, 0)
print('[INFO] SNR-scaled component noises are available.')
existing_train = load_existing_stream('CombinedNoise_Train.pkl')
existing_test = load_existing_stream('CombinedNoise_Test.pkl')
print('[INFO] existing CombinedNoise_Train:', existing_train.shape)
print('[INFO] existing CombinedNoise_Test:', existing_test.shape)

[INFO] SNR-scaled component noises are available.
[INFO] existing CombinedNoise_Train: (650000,)
[INFO] existing CombinedNoise_Test: (650000,)


In [4]:
def power(x, axis=-1, eps=1e-12):
    x = np.asarray(x)
    return np.mean(x * x, axis=axis) + eps

def snr_db(signal, noise, axis=-1):
    ps = power(signal, axis=axis)
    pn = power(noise, axis=axis)
    return 10.0 * np.log10(ps / pn)

def split_stream_to_beats(stream_1d, n_beats, samples=SAMPLES):
    stream_1d = np.asarray(stream_1d).reshape(-1)
    needed = n_beats * samples
    if stream_1d.shape[0] < needed:
        raise ValueError(f'stream too short: {stream_1d.shape[0]} < {needed}')
    x = stream_1d[:needed].reshape(n_beats, samples)
    return x

def summarize_snr(name, clean, noisy):
    noise = noisy - clean
    s = snr_db(clean, noise, axis=1)
    print(f'[{name}] SNR(dB): mean={s.mean():.2f} std={s.std():.2f} min={s.min():.2f} max={s.max():.2f}')
    return s

## Protocols to Compare

We generate **noisy signals** using the same clean beats (QTDB) and NSTDB-derived noises.

- **P0: Existing CENSD stream** (what your pipeline already saved as CombinedNoise_Train/Test)
- **P1: Fixed-SNR Fixed-mix** (BW+EM+MA at a single SNR per component; stationary across beats)
- **P2: RMN-style (sample-wise random mix)** (random subset of {BW,EM,MA} per beat + random SNR per beat; stationary *within* beat, no temporal structure)
- **P3: CENSD re-synthesis (chunk-wise, component-wise random SNR)** (time-varying across beat index; matches the intended CENSD design)

In [18]:
@dataclass
class ProtocolConfig:
    name: str
    seed: int = DEFAULT_SEED
    chunk_size: int = 10000  # for stream-based protocols

def build_fixed_stream(n_total, snr_db_level=0, channel=0, seed=DEFAULT_SEED):
    # All-three mixed at fixed per-component SNR level
    bw = load_noise_component('BW', snr_db_level)[channel]
    em = load_noise_component('EM', snr_db_level)[channel]
    ma = load_noise_component('MA', snr_db_level)[channel]
    mixed = (bw + em + ma).astype(np.float32)
    if mixed.shape[0] < n_total:
        reps = int(np.ceil(n_total / mixed.shape[0]))
        mixed = np.tile(mixed, reps)
    return mixed[:n_total]

def build_rmn_stream(n_total, channel=0, seed=DEFAULT_SEED):
    # Beat-wise random subset + random SNR (sample-wise). Implemented as a stream for easy slicing.
    rng = np.random.default_rng(seed)
    n_beats = n_total // SAMPLES
    out = np.zeros((n_beats, SAMPLES), dtype=np.float32)
    noise_idx = 0
    # Preload component noises for all SNR levels for speed
    bw_all = {s: load_noise_component('BW', s)[channel] for s in SNR_LEVELS}
    em_all = {s: load_noise_component('EM', s)[channel] for s in SNR_LEVELS}
    ma_all = {s: load_noise_component('MA', s)[channel] for s in SNR_LEVELS}
    L = bw_all[SNR_LEVELS[0]].shape[0]
    for i in range(n_beats):
        # choose subset among 8 combos, excluding all-zero with small prob (can be tuned)
        combo = rng.integers(0, 8)  # 0..7 bitmask for (BW,EM,MA)
        bw_on = (combo & 1) != 0
        em_on = (combo & 2) != 0
        ma_on = (combo & 4) != 0
        # sample-wise SNR: pick one per component (keeps per-beat stationary, but component-wise can differ)
        bw_s = rng.choice(SNR_LEVELS)
        em_s = rng.choice(SNR_LEVELS)
        ma_s = rng.choice(SNR_LEVELS)
        start = noise_idx
        end = start + SAMPLES
        if end > L:
            noise_idx = 0
            start = 0
            end = SAMPLES
        seg = np.zeros(SAMPLES, dtype=np.float32)
        if bw_on:
            seg += bw_all[bw_s][start:end]
        if em_on:
            seg += em_all[em_s][start:end]
        if ma_on:
            seg += ma_all[ma_s][start:end]
        out[i] = seg
        noise_idx += SAMPLES
    return out.reshape(-1)[:n_total]

def build_censd_stream(n_total, channel=0, chunk_size=10000, seed=DEFAULT_SEED):
    # Chunk-wise random component-wise SNR; resembles qtdb.ipynb generate_combined_noise()
    rng = np.random.default_rng(seed)
    out = np.zeros(n_total, dtype=np.float32)
    # Preload component noises for all SNR levels
    bw_all = {s: load_noise_component('BW', s)[channel] for s in SNR_LEVELS}
    em_all = {s: load_noise_component('EM', s)[channel] for s in SNR_LEVELS}
    ma_all = {s: load_noise_component('MA', s)[channel] for s in SNR_LEVELS}
    L = bw_all[SNR_LEVELS[0]].shape[0]
    if L < n_total:
        # tile sources if needed (rare for our n_total)
        reps = int(np.ceil(n_total / L))
        for d in (bw_all, em_all, ma_all):
            for k in list(d.keys()):
                d[k] = np.tile(d[k], reps)
        L = bw_all[SNR_LEVELS[0]].shape[0]
    n_chunks = int(np.ceil(n_total / chunk_size))
    for ci in range(n_chunks):
        start = ci * chunk_size
        end = min(n_total, start + chunk_size)
        bw_s = rng.choice(SNR_LEVELS)
        em_s = rng.choice(SNR_LEVELS)
        ma_s = rng.choice(SNR_LEVELS)
        out[start:end] = (
            bw_all[bw_s][start:end] +
            em_all[em_s][start:end] +
            ma_all[ma_s][start:end]
        ).astype(np.float32)
    return out


In [21]:
# --- Controlled shift evaluation: ID vs OOD-SNR vs OOD-Composition ---
EVAL_SEED = 123
CHANNEL = 0

# choose an ID train SNR support and an OOD test support (non-overlapping)
ID_SNR_CHOICES = np.array([-3, 0, 3, 6], dtype=np.int32)
OOD_SNR_CHOICES = np.array([-6, 12, 18], dtype=np.int32)

# composition holdout: exclude multi-component mixes in test, for example
# mask bits: 1=BW, 2=EM, 4=MA
HOLDOUT_COMBOS = {3, 5, 6, 7}  # any 2-or-3 component mix excluded from test (edit as needed)

def _quantile_summary(x, qs=(0.0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0)):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {'n': 0}
    qv = np.quantile(x, qs)
    out = {'n': int(x.size)}
    for q, v in zip(qs, qv):
        out[f"q{int(round(q*100)):02d}"] = float(v)
    return out

def scale_noise_to_target_snr(clean_beat, noise_beat, target_snr_db):
    clean_beat = np.asarray(clean_beat, dtype=np.float32)
    noise_beat = np.asarray(noise_beat, dtype=np.float32)
    px = float(np.mean(np.square(clean_beat)))
    pn = float(np.mean(np.square(noise_beat)))
    if pn <= 1e-12 or px <= 1e-12:
        return np.zeros_like(noise_beat, dtype=np.float32)
    target_pn = px / (10.0 ** (float(target_snr_db) / 10.0))
    scale = float(np.sqrt(target_pn / pn))
    return (noise_beat * scale).astype(np.float32)

def _preload_components(channel=0):
    bw_all = {s: load_noise_component('BW', s)[channel].astype(np.float32) for s in SNR_LEVELS}
    em_all = {s: load_noise_component('EM', s)[channel].astype(np.float32) for s in SNR_LEVELS}
    ma_all = {s: load_noise_component('MA', s)[channel].astype(np.float32) for s in SNR_LEVELS}
    L = bw_all[SNR_LEVELS[0]].shape[0]
    return bw_all, em_all, ma_all, L

def build_rmn_fair_beats(clean_beats_2d, channel=0, seed=DEFAULT_SEED,
                         target_snr_choices=SNR_LEVELS, combo_exclude=None):
    rng = np.random.default_rng(seed)
    n_beats = int(clean_beats_2d.shape[0])
    out_noise = np.zeros_like(clean_beats_2d, dtype=np.float32)
    meta = {
        'combo_mask': np.zeros(n_beats, dtype=np.int32),
        'target_snr_db': np.zeros(n_beats, dtype=np.float32),
    }
    bw_all, em_all, ma_all, L = _preload_components(channel=channel)
    noise_idx = 0
    # exclude combo=0 (no noise) to avoid infinite/degenerate SNR
    allowed = list(range(1, 8))
    if combo_exclude is not None:
        allowed = [m for m in allowed if m not in set(combo_exclude)]
        if not allowed:
            raise ValueError('combo_exclude removed all combos')
    for i in range(n_beats):
        combo = int(rng.choice(allowed))
        bw_on = (combo & 1) != 0
        em_on = (combo & 2) != 0
        ma_on = (combo & 4) != 0
        start = noise_idx
        end = start + SAMPLES
        if end > L:
            noise_idx = 0
            start = 0
            end = SAMPLES
        seg = np.zeros(SAMPLES, dtype=np.float32)
        bw_s = int(rng.choice(SNR_LEVELS))
        em_s = int(rng.choice(SNR_LEVELS))
        ma_s = int(rng.choice(SNR_LEVELS))
        if bw_on:
            seg += bw_all[bw_s][start:end]
        if em_on:
            seg += em_all[em_s][start:end]
        if ma_on:
            seg += ma_all[ma_s][start:end]
        target = float(rng.choice(target_snr_choices))
        seg = scale_noise_to_target_snr(clean_beats_2d[i], seg, target_snr_db=target)
        out_noise[i] = seg
        meta['combo_mask'][i] = combo
        meta['target_snr_db'][i] = target
        noise_idx += SAMPLES
    return out_noise, meta

def evaluate_protocol_on_beats(name, clean_beats_2d, noise_beats_2d):
    noisy = clean_beats_2d + noise_beats_2d
    snr_values = np.array([snr_db(clean_beats_2d[i], noise_beats_2d[i]) for i in range(clean_beats_2d.shape[0])], dtype=np.float32)
    return {
        'snr_db': snr_values,
        'snr_summary': summarize_snr(name, clean_beats_2d, noisy),
        'snr_quantiles': _quantile_summary(snr_values),
        'noisy_power': float(np.mean(np.square(noisy))),
    }

def print_eval(title, out, meta=None):
    print(f"\n=== {title} ===")
    print('SNR summary:', out['snr_summary'])
    print('SNR quantiles:', out['snr_quantiles'])
    if meta is not None:
        keys = [k for k in meta.keys() if k in ('combo_mask','target_snr_db')]
        for k in keys:
            v = meta[k]
            if k == 'combo_mask':
                unique, counts = np.unique(v, return_counts=True)
                print('combo_mask counts:', dict(zip(unique.tolist(), counts.tolist())))
            else:
                print(f"{k} quantiles:", _quantile_summary(v, qs=(0.0,0.1,0.5,0.9,1.0)))

# Use the safe cap if it exists, else fall back
N = int(min(N_BEATS_SAFE if 'N_BEATS_SAFE' in globals() else 1200, clean_beats.shape[0]))
clean_eval = clean_beats[:N].astype(np.float32)

# Protocol 1: Fixed (stationary) at 0 dB components
fixed_stream = build_fixed_stream(n_total=N*SAMPLES, snr_db_level=0, channel=CHANNEL, seed=EVAL_SEED)
fixed_noise = split_stream_to_beats(fixed_stream, n_beats=N)
fixed_out = evaluate_protocol_on_beats('Fixed(all3 @ 0 dB)', clean_eval, fixed_noise)
print_eval('Fixed(all3 @ 0 dB)', fixed_out)

# Protocol 2: CENSD chunk-wise component SNR (stream -> beats)
censd_stream = build_censd_stream(n_total=N*SAMPLES, channel=CHANNEL, chunk_size=10000, seed=EVAL_SEED)
censd_noise = split_stream_to_beats(censd_stream, n_beats=N)
censd_meta = {'chunk_size': 10000}
censd_out = evaluate_protocol_on_beats('CENSD(chunk-wise component SNR)', clean_eval, censd_noise)
print_eval('CENSD(chunk-wise component SNR)', censd_out, censd_meta)

# Protocol 3a: RMN-fair (ID support)
rmn_id_noise, rmn_id_meta = build_rmn_fair_beats(clean_eval, channel=CHANNEL, seed=EVAL_SEED,
                                                target_snr_choices=ID_SNR_CHOICES, combo_exclude=None)
rmn_id_out = evaluate_protocol_on_beats('RMN-fair(ID SNR support)', clean_eval, rmn_id_noise)
print_eval('RMN-fair(ID SNR support)', rmn_id_out, rmn_id_meta)

# Protocol 3b: RMN-fair (OOD-SNR support)
rmn_ood_snr_noise, rmn_ood_snr_meta = build_rmn_fair_beats(clean_eval, channel=CHANNEL, seed=EVAL_SEED+1,
                                                          target_snr_choices=OOD_SNR_CHOICES, combo_exclude=None)
rmn_ood_snr_out = evaluate_protocol_on_beats('RMN-fair(OOD-SNR support)', clean_eval, rmn_ood_snr_noise)
print_eval('RMN-fair(OOD-SNR support)', rmn_ood_snr_out, rmn_ood_snr_meta)

# Protocol 3c: RMN-fair (OOD-composition: exclude holdout combos)
rmn_ood_comp_noise, rmn_ood_comp_meta = build_rmn_fair_beats(clean_eval, channel=CHANNEL, seed=EVAL_SEED+2,
                                                            target_snr_choices=ID_SNR_CHOICES, combo_exclude=HOLDOUT_COMBOS)
rmn_ood_comp_out = evaluate_protocol_on_beats('RMN-fair(OOD-composition holdout)', clean_eval, rmn_ood_comp_noise)
print_eval('RMN-fair(OOD-composition holdout)', rmn_ood_comp_out, rmn_ood_comp_meta)

[Fixed(all3 @ 0 dB)] SNR(dB): mean=-5.40 std=3.93 min=-20.69 max=6.80

=== Fixed(all3 @ 0 dB) ===
SNR summary: [-7.0728707 -3.7513578 -9.222651  ... -8.980217  -8.655554  -5.3093233]
SNR quantiles: {'n': 1200, 'q00': -20.68926239013672, 'q05': -11.8980064868927, 'q10': -10.202720069885254, 'q25': -7.876296520233154, 'q50': -5.309858798980713, 'q75': -2.894474983215332, 'q90': -0.4172330647706918, 'q95': 1.1509614348411559, 'q100': 6.797297477722168}
[CENSD(chunk-wise component SNR)] SNR(dB): mean=-4.14 std=5.82 min=-25.49 max=14.34

=== CENSD(chunk-wise component SNR) ===
SNR summary: [-11.520864   -9.496814   -9.983939  ...  -0.6561443   3.4461036
   6.976593 ]
SNR quantiles: {'n': 1200, 'q00': -25.494672775268555, 'q05': -12.76306438446045, 'q10': -10.816939640045167, 'q25': -8.000559329986572, 'q50': -4.9040186405181885, 'q75': -0.6962069422006607, 'q90': 3.861890006065372, 'q95': 7.029385161399841, 'q100': 14.340343475341797}
[RMN-fair(ID SNR support)] SNR(dB): mean=1.36 std=3.34 m

In [8]:
# ---- Safe benchmark block (caps N_BEATS to available stream length) ----
max_beats_from_existing = int(existing_train.reshape(-1).shape[0] // SAMPLES)
N_BEATS_SAFE = min(1200, max_beats_from_existing, int(clean_beats.shape[0]))
clean = clean_beats[:N_BEATS_SAFE]
n_total = N_BEATS_SAFE * SAMPLES
print(f"[INFO] Benchmark beats: {N_BEATS_SAFE} (n_total={n_total})")

# P0: existing CENSD streams (1D arrays (650000,) in this pipeline)
p0_train_stream = existing_train.reshape(-1)[:n_total]
p0_noisy = clean + split_stream_to_beats(p0_train_stream, N_BEATS_SAFE)

# P1: fixed SNR (0 dB)
p1_stream = build_fixed_stream(n_total=n_total, snr_db_level=0, channel=0)
p1_noisy = clean + split_stream_to_beats(p1_stream, N_BEATS_SAFE)

# P2: RMN-style (sample-wise random mix + random scaling)
p2_stream = build_rmn_stream(n_total=n_total, channel=0, seed=DEFAULT_SEED)
p2_noisy = clean + split_stream_to_beats(p2_stream, N_BEATS_SAFE)

# P3: CENSD re-synthesis (chunk-wise random component SNR)
p3_stream = build_censd_stream(n_total=n_total, channel=0, chunk_size=10000, seed=DEFAULT_SEED)
p3_noisy = clean + split_stream_to_beats(p3_stream, N_BEATS_SAFE)

s0 = summarize_snr('P0 existing-CENSD', clean, p0_noisy)
s1 = summarize_snr('P1 fixed-mix fixed-SNR(0dB)', clean, p1_noisy)
s2 = summarize_snr('P2 RMN-style sample-wise', clean, p2_noisy)
s3 = summarize_snr('P3 CENSD re-synthesis', clean, p3_noisy)

def _snr_stats(x):
    return float(np.mean(x)), float(np.std(x)), float(np.min(x)), float(np.max(x))

print('[INFO] SNR stats: mean, std, min, max')
print('P0 existing-CENSD        :', _snr_stats(s0))
print('P1 fixed (0 dB target)   :', _snr_stats(s1))
print('P2 RMN-style             :', _snr_stats(s2))
print('P3 CENSD re-synthesis    :', _snr_stats(s3))

[INFO] Benchmark beats: 1200 (n_total=614400)
[P0 existing-CENSD] SNR(dB): mean=-2.58 std=6.62 min=-25.43 max=20.39
[P1 fixed-mix fixed-SNR(0dB)] SNR(dB): mean=-5.40 std=3.93 min=-20.69 max=6.80
[P2 RMN-style sample-wise] SNR(dB): mean=14.73 std=36.55 min=-20.94 max=108.37
[P3 CENSD re-synthesis] SNR(dB): mean=-3.41 std=6.41 min=-22.74 max=23.13
[INFO] SNR stats: mean, std, min, max
P0 existing-CENSD        : (-2.581485931464823, 6.623008275998664, -25.42924028674166, 20.389519305534918)
P1 fixed (0 dB target)   : (-5.403697967529297, 3.9277665615081787, -20.68926239013672, 6.797297477722168)
P2 RMN-style             : (14.731523513793945, 36.54560470581055, -20.943777084350586, 108.37187194824219)
P3 CENSD re-synthesis    : (-3.4111523628234863, 6.410065650939941, -22.741071701049805, 23.126802444458008)


In [7]:
# Build small benchmark datasets for diagnostics (fast)
N_BEATS = 4000  # keep this moderate for quick runs
clean = clean_beats[:N_BEATS]
n_total = N_BEATS * SAMPLES

# P0: existing CENSD streams (already saved). These are 1D arrays (650000,) in your pipeline.
p0_train_stream = existing_train.reshape(-1)[:n_total]
p0_test_stream = existing_test.reshape(-1)[:n_total]

# P1: fixed SNR (choose one representative, e.g., 0 dB)
p1_stream = build_fixed_stream(n_total=n_total, snr_db_level=0, channel=0)

# P2: RMN-style sample-wise random mix
p2_stream = build_rmn_stream(n_total=n_total, channel=0, seed=DEFAULT_SEED)

# P3: CENSD re-synthesis (chunk-wise random component SNR)
p3_stream = build_censd_stream(n_total=n_total, channel=0, chunk_size=10000, seed=DEFAULT_SEED)

# Slice into beats and create noisy signals
p0_noisy = clean + split_stream_to_beats(p0_train_stream, N_BEATS)
p1_noisy = clean + split_stream_to_beats(p1_stream, N_BEATS)
p2_noisy = clean + split_stream_to_beats(p2_stream, N_BEATS)
p3_noisy = clean + split_stream_to_beats(p3_stream, N_BEATS)

s0 = summarize_snr('P0 existing-CENSD', clean, p0_noisy)
s1 = summarize_snr('P1 fixed-mix fixed-SNR(0dB)', clean, p1_noisy)
s2 = summarize_snr('P2 RMN-style sample-wise', clean, p2_noisy)
s3 = summarize_snr('P3 CENSD re-synthesis', clean, p3_noisy)

ValueError: stream too short: 650000 < 2048000

In [9]:
def temporal_variation_metric(snr_series):
    # Larger means more rapidly changing conditions across sequential beats
    diffs = np.diff(snr_series)
    return float(np.mean(np.abs(diffs))), float(np.std(diffs))

m0 = temporal_variation_metric(s0)
m1 = temporal_variation_metric(s1)
m2 = temporal_variation_metric(s2)
m3 = temporal_variation_metric(s3)

print('[Temporal | mean|ΔSNR|, std(ΔSNR)]')
print('P0 existing-CENSD:', m0)
print('P1 fixed:', m1)
print('P2 RMN:', m2)
print('P3 CENSD re-synth:', m3)

[Temporal | mean|ΔSNR|, std(ΔSNR)]
P0 existing-CENSD: (3.9366708697169748, 5.212107800357772)
P1 fixed: (3.146120309829712, 3.9965124130249023)
P2 RMN: (30.63328742980957, 51.252315521240234)
P3 CENSD re-synth: (3.6257591247558594, 4.663049697875977)


## Next: Robustness Benchmarks (Hooks)

The next step for the paper-facing claim is to train/evaluate denoisers under:
- ID: same protocol train/test
- OOD SNR shift: train on mid SNR, test on low SNR
- OOD composition shift: train without certain (BW,EM,MA) dominance, test with it
- Worst-case metrics (tail)

This notebook intentionally keeps training out-of-scope for now (it can be slow). Once you pick 2–3 architectures to benchmark, we can add compact training/eval cells that reuse your existing model builders and save results tables.

## External Shift (SimEMG) — Reference-Based Robustness Evidence

이 섹션은 **SimEMG의 reference(near-clean) 채널**을 이용해서,
외부 도메인에서의 성능을 **정량 지표 + tail(worst-k%)**로 요약합니다.

핵심 포인트:
- SimEMG는 QTDB+NSTDB로 합성한 노이즈와 분포가 다를 수 있음(센서/전극/EMG 양상).
- 따라서 여기서의 성능은 “합성 프로토콜에 대한 overfitting”이 아니라 **외부 일반화** 근거가 됩니다.

아래 코드는 1) `.mat`에서 2채널을 읽고, 2) HF 에너지로 clean/noisy 채널을 자동 추정하고,
3) 512-sample windows로 잘라 지표를 계산합니다.

In [22]:
from pathlib import Path
import re
import pandas as pd
from scipy.io import loadmat

SIMEMG_DIR = Path('data') / 'SimEMG'
print('[INFO] SIMEMG_DIR:', SIMEMG_DIR, 'exists=', SIMEMG_DIR.exists())

def _parse_simemg_meta(path: Path):
    # Example: P10_1_Ag-AgCl.mat, P10_1_ORB.mat, P10_1_lead I.mat
    m = re.match(r'^P(?P<subject>\d+)_?(?P<trial>\d+)?_(?P<modality>.+)\.mat$', path.name)
    if not m:
        return {'subject': None, 'trial': None, 'modality': None}
    d = m.groupdict()
    return {
        'subject': int(d['subject']) if d['subject'] else None,
        'trial': int(d['trial']) if d['trial'] else None,
        'modality': d['modality'],
    }

def load_simemg_two_channel(path: Path):
    """Load a SimEMG .mat and return float32 array shape (N,2)."""
    d = loadmat(path)
    arrays = []
    for k, v in d.items():
        if k.startswith('__'):
            continue
        if isinstance(v, np.ndarray) and v.ndim == 2:
            arrays.append((k, v))
    if not arrays:
        raise ValueError(f'No 2D array found in {path.name}')
    # pick the first 2D array with 2 channels (either (N,2) or (2,N))
    key, a = None, None
    for k, v in arrays:
        if v.shape[1] == 2 or v.shape[0] == 2:
            key, a = k, v
            break
    if a is None:
        # fallback: largest 2D array
        key, a = max(arrays, key=lambda kv: kv[1].size)
    a = np.asarray(a)
    if a.shape[0] == 2 and a.shape[1] != 2:
        a = a.T
    if a.shape[1] != 2:
        raise ValueError(f'Expected 2 channels, got shape {a.shape} from key={key}')
    return a.astype(np.float32)

def infer_clean_noisy(two_ch: np.ndarray):
    """Infer which channel is noisy via high-frequency proxy energy (diff energy)."""
    x0 = two_ch[:, 0]
    x1 = two_ch[:, 1]
    hf0 = float(np.mean(np.square(np.diff(x0))))
    hf1 = float(np.mean(np.square(np.diff(x1))))
    # higher HF => more likely noisy
    if hf0 >= hf1:
        noisy, clean = x0, x1
        noisy_idx = 0
    else:
        noisy, clean = x1, x0
        noisy_idx = 1
    return clean.astype(np.float32), noisy.astype(np.float32), {'hf0': hf0, 'hf1': hf1, 'noisy_idx': noisy_idx}

def window_signal_1d(x: np.ndarray, win=SAMPLES, hop=SAMPLES):
    x = np.asarray(x).reshape(-1)
    n = x.shape[0]
    if n < win:
        return np.zeros((0, win), dtype=np.float32)
    starts = np.arange(0, n - win + 1, hop, dtype=np.int64)
    out = np.stack([x[s:s+win] for s in starts], axis=0)
    return out.astype(np.float32)

def rmse(a, b, axis=-1):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return np.sqrt(np.mean((a - b) ** 2, axis=axis) + 1e-12)

def prd(a, b, axis=-1):
    # percent root-mean-square difference
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    num = np.sqrt(np.sum((a - b) ** 2, axis=axis))
    den = np.sqrt(np.sum(a ** 2, axis=axis) + 1e-12)
    return 100.0 * (num / (den + 1e-12))

def snr_db_windows(clean_w, noisy_w):
    noise_w = noisy_w - clean_w
    return snr_db(clean_w, noise_w, axis=1).astype(np.float32)

print('[INFO] Ready: SimEMG loaders + metrics')

/tmp/ipykernel_2539042/4547805.py:4: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.io import loadmat


[INFO] SIMEMG_DIR: data/SimEMG exists= True
[INFO] Ready: SimEMG loaders + metrics


In [23]:
# Evaluate SimEMG: per-file + overall + tail (worst-k%)
def evaluate_simemg_file(path: Path, win=SAMPLES, hop=SAMPLES, tail_frac=0.1, denoise_fn=None):
    meta = _parse_simemg_meta(path)
    two = load_simemg_two_channel(path)
    clean_1d, noisy_1d, info = infer_clean_noisy(two)
    clean_w = window_signal_1d(clean_1d, win=win, hop=hop)
    noisy_w = window_signal_1d(noisy_1d, win=win, hop=hop)
    n_w = int(clean_w.shape[0])
    if n_w == 0:
        return {**meta, 'path': str(path), 'n_windows': 0}
    snr_in = snr_db_windows(clean_w, noisy_w)
    # optional denoiser hook: denoise_fn(noisy_windows)->denoised_windows
    if denoise_fn is None:
        den_w = None
        snr_out = None
        prd_out = None
        rmse_out = None
    else:
        den_w = np.asarray(denoise_fn(noisy_w), dtype=np.float32)
        if den_w.shape != noisy_w.shape:
            raise ValueError(f'denoise_fn must return shape {noisy_w.shape}, got {den_w.shape}')
        snr_out = snr_db_windows(clean_w, den_w)
        prd_out = prd(clean_w, den_w, axis=1)
        rmse_out = rmse(clean_w, den_w, axis=1)
    # tail windows: worst by input SNR
    k = max(1, int(np.ceil(tail_frac * n_w)))
    worst_idx = np.argsort(snr_in)[:k]
    out = {
        **meta,
        'path': str(path),
        'n_windows': n_w,
        'hf0': info['hf0'],
        'hf1': info['hf1'],
        'noisy_idx': info['noisy_idx'],
        'snr_in_mean': float(np.mean(snr_in)),
        'snr_in_p05': float(np.quantile(snr_in, 0.05)),
        'snr_in_p50': float(np.quantile(snr_in, 0.50)),
        'snr_in_p95': float(np.quantile(snr_in, 0.95)),
        'snr_in_worst_mean': float(np.mean(snr_in[worst_idx])),
    }
    if snr_out is not None:
        out.update({
            'snr_out_mean': float(np.mean(snr_out)),
            'snr_out_p05': float(np.quantile(snr_out, 0.05)),
            'snr_out_p50': float(np.quantile(snr_out, 0.50)),
            'snr_out_p95': float(np.quantile(snr_out, 0.95)),
            'snr_out_worst_mean': float(np.mean(snr_out[worst_idx])),
            'prd_out_mean': float(np.mean(prd_out)),
            'rmse_out_mean': float(np.mean(rmse_out)),
        })
    return out

def evaluate_simemg_dataset(simemg_dir=SIMEMG_DIR, max_files=None, tail_frac=0.1, denoise_fn=None):
    paths = sorted(simemg_dir.glob('*.mat'))
    if max_files is not None:
        paths = paths[:int(max_files)]
    rows = []
    for p in paths:
        try:
            rows.append(evaluate_simemg_file(p, tail_frac=tail_frac, denoise_fn=denoise_fn))
        except Exception as e:
            rows.append({**_parse_simemg_meta(p), 'path': str(p), 'n_windows': 0, 'error': repr(e)})
    df = pd.DataFrame(rows)
    return df

# Run baseline external stats (no denoiser): this is the external shift characterization
df_sim = evaluate_simemg_dataset(max_files=30, tail_frac=0.1, denoise_fn=None)
print('[INFO] SimEMG rows:', df_sim.shape)
display(df_sim.head())

# Summaries (ignore failed rows)
ok = df_sim[df_sim['n_windows'] > 0].copy()
print('[INFO] OK files:', ok.shape[0], ' / total:', df_sim.shape[0])
if ok.shape[0] > 0:
    print('SNR_in mean over files:', ok['snr_in_mean'].describe())
    print('SNR_in tail(worst10%) mean over files:', ok['snr_in_worst_mean'].describe())
    by_mod = ok.groupby('modality', dropna=False).agg({
        'snr_in_mean': ['mean','std','min','max'],
        'snr_in_worst_mean': ['mean','std','min','max'],
        'n_windows': 'sum',
    })
    display(by_mod)

[INFO] SimEMG rows: (30, 13)


,subject,trial,modality,path,n_windows,hf0,hf1,noisy_idx,snr_in_mean,snr_in_p05,snr_in_p50,snr_in_p95,snr_in_worst_mean
0,10,1,Ag-AgCl,data/SimEMG/P10_1_Ag-AgCl.mat,29,14.305972,34.711185,1,10.068227,8.810514,9.940697,12.099732,8.510785
1,10,1,ORB,data/SimEMG/P10_1_ORB.mat,29,14.305972,32.399597,1,10.510370,9.304970,10.240594,12.733941,8.840220
2,10,1,lead I,data/SimEMG/P10_1_lead I.mat,29,14.305972,31.077011,1,10.773285,9.325918,10.600233,12.672068,9.288623
3,11,1,Ag-AgCl,data/SimEMG/P11_1_Ag-AgCl.mat,29,35.316605,72.153770,1,12.339047,9.408503,12.090478,15.953773,8.003875
4,11,1,ORB,data/SimEMG/P11_1_ORB.mat,29,35.316605,72.758629,1,11.666748,5.856528,11.997740,15.652749,3.535919


[INFO] OK files: 30  / total: 30
SNR_in mean over files: count    30.000000
mean     11.839566
std       5.217278
min       1.358932
25%      10.127594
50%      12.254251
75%      15.121760
max      20.769703
Name: snr_in_mean, dtype: float64
SNR_in tail(worst10%) mean over files: count    30.000000
mean      9.091250
std       5.359384
min      -2.507474
25%       4.505797
50%       9.604025
75%      13.065259
max      19.261187
Name: snr_in_worst_mean, dtype: float64


snr_in_mean                                 snr_in_worst_mean  \
                mean       std        min        max              mean   
modality                                                                 
Ag-AgCl    10.257793  5.945705   1.358932  20.769703          7.925550   
ORB        10.617927  5.105857   2.510763  19.202719          6.869364   
lead I     15.130222  2.768932  10.773285  19.340813         12.984757   

                                        n_windows  
               std       min        max       sum  
modality                                           
Ag-AgCl   6.236903 -2.507474  19.261187       319  
ORB       4.410959  0.291174  13.036588       290  
lead I    2.918013  9.288623  17.576284       261

## External Shift Results (SimEMG) — Using Saved CSV (CENSD-trained: `0221_fixed`)


방금 확인된 전제: `0221_fixed` 세팅은 **CENSD로 학습된 모델 가중치/평가 결과**입니다.


이 섹션은 이미 저장된 CSV(`evaluation_results_simemg/`)를 로드해서,


- **Mean 성능(전체 평균)**: noisy 대비 denoised의 Δ (예: $\Delta\mathrm{SNR}=\mathrm{SNR}_{out}-\mathrm{SNR}_{in}$)


- **Tail 성능(worst 10%)**: 입력 SNR이 가장 낮은 구간에서의 평균 SNR_out 및 Δ


을 한 번에 요약합니다.


주의: 위에서 노트북에서 계산한 `snr_in_mean`(512-window 기반, 채널 자동 판별)과 CSV의 `snr_noisy_db_mean`은 **윈도우/세그먼트 정의, 채널 판별 방식, 전처리** 차이로 값이 달라질 수 있습니다. 여기서는 **같은 CSV 내부에서의 상대 비교와 tail 경향**에 집중합니다.

In [24]:
# --- Load saved SimEMG evaluation CSVs (CENSD-trained setting: 0221_fixed) ---
from pathlib import Path
import pandas as pd
import numpy as np

SIMEMG_RES_DIR = Path('evaluation_results_simemg')
paths = {
    'overall_wide': SIMEMG_RES_DIR / 'simemg_0221_fixed_overall_summary_wide.csv',
    '10s_overall_wide': SIMEMG_RES_DIR / 'simemg_0221_fixed_10s_overall_summary_wide.csv',
    'segment_metrics': SIMEMG_RES_DIR / 'simemg_0221_fixed_segment_metrics.csv',
    '10s_sequence_metrics': SIMEMG_RES_DIR / 'simemg_0221_fixed_10s_sequence_metrics.csv',
    'fewshot_summary': SIMEMG_RES_DIR / 'fewshot_dual_freqdae' / 'fewshot_summary_holdoutSubj2.csv',
}
for k,p in paths.items():
    print(f"[INFO] {k}: {p} exists={p.exists()}")

def _first_col(df: pd.DataFrame, base: str):
    cols = [c for c in df.columns if c == base or c.startswith(base)]
    if not cols:
        raise KeyError(f"Missing column base '{base}'. Available head: {list(df.columns)[:12]}")
    return cols[0]

def _add_deltas(df: pd.DataFrame):
    snr_in_col = _first_col(df, 'snr_noisy_db_mean')
    snr_out_col = _first_col(df, 'snr_denoised_db_mean')
    rmse_in_col = _first_col(df, 'rmse_noisy_mean')
    rmse_out_col = _first_col(df, 'rmse_denoised_mean')
    prd_in_col = _first_col(df, 'prd_noisy_mean')
    prd_out_col = _first_col(df, 'prd_denoised_mean')
    out = df.copy()
    out['delta_snr_db_mean'] = out[snr_out_col] - out[snr_in_col]
    out['delta_rmse_mean'] = out[rmse_out_col] - out[rmse_in_col]
    out['delta_prd_mean'] = out[prd_out_col] - out[prd_in_col]
    return out

# Overall (file-level aggregated)
df_overall = pd.read_csv(paths['overall_wide'])
df_overall2 = _add_deltas(df_overall)
display(df_overall2.sort_values('delta_snr_db_mean', ascending=False))

# 10s overall (harder slices)
df_10s = pd.read_csv(paths['10s_overall_wide'])
df_10s2 = _add_deltas(df_10s)
display(df_10s2.sort_values('delta_snr_db_mean', ascending=False))

print('[INFO] Overall ΔSNR (dB) summary:')
display(df_overall2[['model','delta_snr_db_mean','snr_noisy_db_mean','snr_denoised_db_mean']].sort_values('delta_snr_db_mean', ascending=False))

print('[INFO] 10s ΔSNR (dB) summary:')
snr_in_10s = _first_col(df_10s2, 'snr_noisy_db_mean')
display(df_10s2[['model','delta_snr_db_mean',snr_in_10s,'snr_denoised_db_mean']].sort_values('delta_snr_db_mean', ascending=False))

# Few-shot summary (Dual_FreqDAE)
if paths['fewshot_summary'].exists():
    df_fs = pd.read_csv(paths['fewshot_summary'])
    display(df_fs)

[INFO] overall_wide: evaluation_results_simemg/simemg_0221_fixed_overall_summary_wide.csv exists=True
[INFO] 10s_overall_wide: evaluation_results_simemg/simemg_0221_fixed_10s_overall_summary_wide.csv exists=True
[INFO] segment_metrics: evaluation_results_simemg/simemg_0221_fixed_segment_metrics.csv exists=True
[INFO] 10s_sequence_metrics: evaluation_results_simemg/simemg_0221_fixed_10s_sequence_metrics.csv exists=True
[INFO] fewshot_summary: evaluation_results_simemg/fewshot_dual_freqdae/fewshot_summary_holdoutSubj2.csv exists=True


,model,rmse_noisy_mean,rmse_denoised_mean,prd_noisy_mean,prd_denoised_mean,cos_noisy_mean,cos_denoised_mean,snr_noisy_db_mean,snr_denoised_db_mean,rmse_noisy_std,rmse_denoised_std,prd_noisy_std,prd_denoised_std,cos_noisy_std,cos_denoised_std,snr_noisy_db_std,snr_denoised_db_std,delta_snr_db_mean,delta_rmse_mean,delta_prd_mean
3,DeepFilter,0.058567,0.071549,49.786262,48.477539,0.88794,0.835869,8.207064,7.058294,0.034765,0.078698,35.290201,21.825544,0.12092,0.218649,6.266034,3.569748,-1.148770,0.012981,-1.308723
6,Transformer_DAE,0.058567,0.093720,49.786262,68.008240,0.88794,0.685803,8.207064,3.774975,0.034765,0.073985,35.290201,20.268098,0.12092,0.242711,6.266034,2.814817,-4.432089,0.035153,18.221978
4,Dual_FreqDAE,0.058567,0.097543,49.786262,71.597366,0.88794,0.657169,8.207064,3.319734,0.034765,0.071406,35.290201,21.305851,0.12092,0.238432,6.266034,2.775777,-4.887330,0.038975,21.811105
0,AttentionSkipDAE,0.058567,0.101378,49.786262,74.285534,0.88794,0.638657,8.207064,2.852637,0.034765,0.070039,35.290201,17.512755,0.12092,0.208875,6.266034,2.247271,-5.354427,0.042811,24.499273
2,DRNN,0.058567,0.122815,49.786262,91.326460,0.88794,0.370096,8.207064,0.862248,0.034765,0.063646,35.290201,11.338361,0.12092,0.223462,6.266034,1.168665,-7.344816,0.064248,41.540199
1,CNN_DAE,0.058567,0.131916,49.786262,97.537178,0.88794,0.277116,8.207064,0.275110,0.034765,0.070030,35.290201,11.040054,0.12092,0.236374,6.266034,1.023051,-7.931954,0.073349,47.750916
5,FCN_DAE,0.058567,0.132800,49.786262,98.264669,0.88794,0.259212,8.207064,0.208780,0.034765,0.069978,35.290201,10.969747,0.12092,0.227452,6.266034,1.008595,-7.998284,0.074232,48.478407


,model,rmse_noisy_mean,rmse_denoised_mean,prd_noisy_mean,prd_denoised_mean,cos_noisy_mean,cos_denoised_mean,snr_noisy_db_mean,snr_noisy_db_mean.1,snr_denoised_db_mean,...,prd_noisy_std,prd_denoised_std,cos_noisy_std,cos_denoised_std,snr_noisy_db_std,snr_noisy_db_std.1,snr_denoised_db_std,delta_snr_db_mean,delta_rmse_mean,delta_prd_mean
3,DeepFilter,0.077488,0.051952,78.016459,51.855114,0.797986,0.866044,2.285521,2.285521,5.798961,...,13.705918,7.554143,0.04961,0.035828,1.496881,1.496881,1.304402,3.513440,-0.025536,-26.161346
6,Transformer_DAE,0.077488,0.071399,78.016459,71.825384,0.797986,0.705019,2.285521,2.285521,2.928011,...,13.705918,7.838327,0.04961,0.066129,1.496881,1.496881,0.982452,0.642490,-0.006089,-6.191076
0,AttentionSkipDAE,0.077488,0.074222,78.016459,74.561293,0.797986,0.670986,2.285521,2.285521,2.592128,...,13.705918,7.178467,0.04961,0.068191,1.496881,1.496881,0.877559,0.306607,-0.003267,-3.455166
4,Dual_FreqDAE,0.077488,0.076655,78.016459,76.969989,0.797986,0.651661,2.285521,2.285521,2.320822,...,13.705918,7.888754,0.04961,0.073669,1.496881,1.496881,0.921986,0.035301,-0.000833,-1.046470
2,DRNN,0.077488,0.088230,78.016459,88.260052,0.797986,0.468461,2.285521,2.285521,1.089332,...,13.705918,2.886873,0.04961,0.056389,1.496881,1.496881,0.284544,-1.196189,0.010741,10.243593
1,CNN_DAE,0.077488,0.093010,78.016459,92.983842,0.797986,0.394544,2.285521,2.285521,0.638138,...,13.705918,3.536038,0.04961,0.060043,1.496881,1.496881,0.332752,-1.647383,0.015521,14.967383
5,FCN_DAE,0.077488,0.097377,78.016459,97.504526,0.797986,0.332535,2.285521,2.285521,0.228336,...,13.705918,4.444941,0.04961,0.057854,1.496881,1.496881,0.392162,-2.057186,0.019889,19.488067


[INFO] Overall ΔSNR (dB) summary:


,model,delta_snr_db_mean,snr_noisy_db_mean,snr_denoised_db_mean
3,DeepFilter,-1.148770,8.207064,7.058294
6,Transformer_DAE,-4.432089,8.207064,3.774975
4,Dual_FreqDAE,-4.887330,8.207064,3.319734
0,AttentionSkipDAE,-5.354427,8.207064,2.852637
2,DRNN,-7.344816,8.207064,0.862248
1,CNN_DAE,-7.931954,8.207064,0.275110
5,FCN_DAE,-7.998284,8.207064,0.208780


[INFO] 10s ΔSNR (dB) summary:


,model,delta_snr_db_mean,snr_noisy_db_mean,snr_denoised_db_mean
3,DeepFilter,3.513440,2.285521,5.798961
6,Transformer_DAE,0.642490,2.285521,2.928011
0,AttentionSkipDAE,0.306607,2.285521,2.592128
4,Dual_FreqDAE,0.035301,2.285521,2.320822
2,DRNN,-1.196189,2.285521,1.089332
1,CNN_DAE,-1.647383,2.285521,0.638138
5,FCN_DAE,-2.057186,2.285521,0.228336


,stage,holdout_subject,modality_predict,n_test_seq,rmse_denoised_mean,prd_denoised_mean,snr_denoised_db_mean,cos_denoised_mean,delta_snr_db_vs_zero,delta_rmse_vs_zero,delta_prd_vs_zero,delta_cos_vs_zero
0,zero_shot,2,NaN,30,0.079972,80.019524,1.949590,0.622834,0.000000,0.000000,0.000000,0.000000
1,few_shot,2,NaN,30,0.060455,60.484397,4.374789,0.807332,2.425199,-0.019517,-19.535127,0.184499


In [27]:
# --- Tail analysis on segments: worst 10% by input SNR ---
seg_path = paths['segment_metrics']
usecols = ['model', 'modality', 'snr_noisy_db', 'snr_denoised_db']
if seg_path.exists():
    # This CSV can be large; load only needed columns for tail stats
    seg = pd.read_csv(seg_path, usecols=usecols)
    seg = seg.dropna(subset=['model', 'snr_noisy_db', 'snr_denoised_db']).copy()
    seg['snr_noisy_db'] = seg['snr_noisy_db'].astype(float)
    seg['snr_denoised_db'] = seg['snr_denoised_db'].astype(float)
    seg['delta_snr_db'] = seg['snr_denoised_db'] - seg['snr_noisy_db']
    print('[INFO] segments loaded:', seg.shape)

    def tail_stats(group: pd.DataFrame, frac=0.10):
        g = group.sort_values('snr_noisy_db', ascending=True)
        k = max(1, int(np.ceil(frac * len(g))))
        tail = g.iloc[:k]
        return pd.Series({
            'n_segments': int(len(g)),
            'snr_in_mean': float(g['snr_noisy_db'].mean()),
            'snr_out_mean': float(g['snr_denoised_db'].mean()),
            'delta_snr_mean': float(g['delta_snr_db'].mean()),
            'snr_in_tail_mean': float(tail['snr_noisy_db'].mean()),
            'snr_out_tail_mean': float(tail['snr_denoised_db'].mean()),
            'delta_snr_tail_mean': float(tail['delta_snr_db'].mean()),
            'snr_in_p05': float(np.quantile(g['snr_noisy_db'], 0.05)),
            'snr_out_p05': float(np.quantile(g['snr_denoised_db'], 0.05)),
        })

    # pandas >= 2.2 warns about group columns; use include_groups when available
    gb_model = seg.groupby('model', dropna=False)
    try:
        tail_by_model = gb_model.apply(tail_stats, include_groups=False).reset_index()
    except TypeError:
        tail_by_model = gb_model.apply(tail_stats).reset_index()

    cols = [
        'model', 'n_segments',
        'snr_in_mean', 'snr_out_mean', 'delta_snr_mean',
        'snr_in_tail_mean', 'snr_out_tail_mean', 'delta_snr_tail_mean',
        'snr_in_p05', 'snr_out_p05',
    ]
    tail_by_model = tail_by_model[cols].sort_values('delta_snr_tail_mean', ascending=False)
    print('[INFO] Tail (worst10%) by model — top 10:')
    display(tail_by_model.head(10))

    # Optional: modality-wise tail summary (top 15 by tail delta)
    gb_mm = seg.groupby(['model', 'modality'], dropna=False)
    try:
        tail_by_model_mod = gb_mm.apply(tail_stats, include_groups=False).reset_index()
    except TypeError:
        tail_by_model_mod = gb_mm.apply(tail_stats).reset_index()
    cols_mm = ['model','modality','n_segments','snr_in_tail_mean','snr_out_tail_mean','delta_snr_tail_mean']
    tail_by_model_mod = tail_by_model_mod[cols_mm].sort_values('delta_snr_tail_mean', ascending=False)
    print('[INFO] Tail (worst10%) by model × modality — top 15:')
    display(tail_by_model_mod.head(15))

[INFO] segments loaded: (16170, 5)
[INFO] Tail (worst10%) by model — top 10:


,model,n_segments,snr_in_mean,snr_out_mean,delta_snr_mean,snr_in_tail_mean,snr_out_tail_mean,delta_snr_tail_mean,snr_in_p05,snr_out_p05
3,DeepFilter,2310.0,8.207064,7.058294,-1.148770,-1.65272,5.488014,7.140735,-1.177881,-0.094154
6,Transformer_DAE,2310.0,8.207064,3.774975,-4.432089,-1.65272,1.674384,3.327105,-1.177881,-0.217226
0,AttentionSkipDAE,2310.0,8.207064,2.852637,-5.354427,-1.65272,1.084127,2.736847,-1.177881,-0.082457
4,Dual_FreqDAE,2310.0,8.207064,3.319734,-4.887330,-1.65272,0.971300,2.624020,-1.177881,-0.500684
2,DRNN,2310.0,8.207064,0.862248,-7.344816,-1.65272,0.219437,1.872157,-1.177881,-0.499745
1,CNN_DAE,2310.0,8.207064,0.275110,-7.931954,-1.65272,-0.064156,1.588564,-1.177881,-1.207800
5,FCN_DAE,2310.0,8.207064,0.208780,-7.998284,-1.65272,-0.183769,1.468952,-1.177881,-1.081978


[INFO] Tail (worst10%) by model × modality — top 15:


,model,modality,n_segments,snr_in_tail_mean,snr_out_tail_mean,delta_snr_tail_mean
10,DeepFilter,ORB,777.0,-2.263037,5.259855,7.522892
9,DeepFilter,Ag-AgCl,777.0,-2.117988,5.392548,7.510536
18,Transformer_DAE,Ag-AgCl,777.0,-2.117988,1.597475,3.715464
19,Transformer_DAE,ORB,777.0,-2.263037,1.356462,3.619500
1,AttentionSkipDAE,ORB,777.0,-2.263037,0.990831,3.253868
11,DeepFilter,lead I,756.0,3.894716,7.130954,3.236238
0,AttentionSkipDAE,Ag-AgCl,777.0,-2.117988,0.913454,3.031442
12,Dual_FreqDAE,Ag-AgCl,777.0,-2.117988,0.856565,2.974553
13,Dual_FreqDAE,ORB,777.0,-2.263037,0.708809,2.971846
7,DRNN,ORB,777.0,-2.263037,0.125268,2.388306


## External Shift: CENSD vs Other Noise Synthesis (Same Architecture, Different Training Sets)


사용자가 원하는 스토리(“다른 노이즈 합성 대비 external shift에서 더 강건”)를 **직접 비교표**로 만들려면,
SimEMG에서 **같은 모델 아키텍처**를 두고 학습 데이터 합성만 바꾼 세팅들을 비교하면 됩니다.


여기서는 아래를 비교 대상으로 둡니다.
- `0221_FIXED` = **CENSD 학습 세팅** (확정)
- `0221_snr0db`, `0221_snr-3db`, `0221_snr3db` = **고정 SNR 기반 합성/학습 세팅(stationary baseline로 해석 가능)**


다음 셀은 SimEMG에 대해 동일한 평가 루틴(채널 판별 + 512-window)으로 `ΔSNR`와 **worst10% tail 개선**을 계산합니다.

In [3]:
# --- Model loader (Keras) for SimEMG inference ---
import numpy as np
import tensorflow as tf
import keras
from deepFilter import dl_models as M

DEFAULT_SIGNAL_SIZE = int(globals().get('SAMPLES', 512))

def build_model_by_name(model_name: str, signal_size: int = DEFAULT_SIGNAL_SIZE):
    if not hasattr(M, model_name):
        raise ValueError(f"Unknown model_name={model_name}. Not found in deepFilter.dl_models")
    fn = getattr(M, model_name)
    if 'signal_size' in fn.__code__.co_varnames:
        model = fn(signal_size=signal_size)
    else:
        model = fn(signal_size)
    return model

def load_model_weights(model_name: str, weights_path: str, signal_size: int = DEFAULT_SIGNAL_SIZE):
    model = build_model_by_name(model_name, signal_size=signal_size)
    dummy = np.zeros((1, signal_size, 1), dtype=np.float32)
    _ = model(dummy, training=False)
    model.load_weights(weights_path)
    return model

def model_denoise_windows(model, noisy_w: np.ndarray, batch_size=256):
    x = noisy_w.astype(np.float32)
    if x.ndim == 2:
        x = x[..., None]
    y = model.predict(x, batch_size=batch_size, verbose=0)
    y = np.asarray(y)
    if y.ndim == 3 and y.shape[-1] == 1:
        y = y[..., 0]
    return y.astype(np.float32)

print('[INFO] Ready: model loader + denoise windows (signal_size=', DEFAULT_SIGNAL_SIZE, ')')

[INFO] Ready: model loader + denoise windows (signal_size= 512 )


In [ ]:
# --- External shift evaluation: compare training settings on SimEMG ---
from pathlib import Path
import re
import pandas as pd
from scipy.io import loadmat

SAMPLES_LOCAL = int(globals().get('SAMPLES', 512))
SIMEMG_DIR_LOCAL = Path(globals().get('SIMEMG_DIR', Path('data') / 'SimEMG'))

def load_simemg_two_channel_local(path: Path):
    d = loadmat(path)
    arrays = []
    for k, v in d.items():
        if k.startswith('__'):
            continue
        if isinstance(v, np.ndarray) and v.ndim == 2:
            arrays.append((k, v))
    if not arrays:
        raise ValueError(f'No 2D array found in {path.name}')
    key, a = None, None
    for k, v in arrays:
        if v.shape[1] == 2 or v.shape[0] == 2:
            key, a = k, v
            break
    if a is None:
        key, a = max(arrays, key=lambda kv: kv[1].size)
    a = np.asarray(a)
    if a.shape[0] == 2 and a.shape[1] != 2:
        a = a.T
    if a.shape[1] != 2:
        raise ValueError(f'Expected 2 channels, got shape {a.shape} from key={key}')
    return a.astype(np.float32)

def infer_clean_noisy_local(two_ch: np.ndarray):
    x0 = two_ch[:, 0]
    x1 = two_ch[:, 1]
    hf0 = float(np.mean(np.square(np.diff(x0))))
    hf1 = float(np.mean(np.square(np.diff(x1))))
    if hf0 >= hf1:
        noisy, clean = x0, x1
        noisy_idx = 0
    else:
        noisy, clean = x1, x0
        noisy_idx = 1
    return clean.astype(np.float32), noisy.astype(np.float32), {'hf0': hf0, 'hf1': hf1, 'noisy_idx': noisy_idx}

def window_signal_1d_local(x: np.ndarray, win=SAMPLES_LOCAL, hop=SAMPLES_LOCAL):
    x = np.asarray(x).reshape(-1)
    n = x.shape[0]
    if n < win:
        return np.zeros((0, win), dtype=np.float32)
    starts = np.arange(0, n - win + 1, hop, dtype=np.int64)
    out = np.stack([x[s:s+win] for s in starts], axis=0)
    return out.astype(np.float32)

def power_local(x, axis=-1, eps=1e-12):
    x = np.asarray(x)
    return np.mean(x * x, axis=axis) + eps

def snr_db_local(signal, noise, axis=-1):
    ps = power_local(signal, axis=axis)
    pn = power_local(noise, axis=axis)
    return 10.0 * np.log10(ps / pn)

def snr_db_windows_local(clean_w, noisy_w):
    noise_w = noisy_w - clean_w
    return snr_db_local(clean_w, noise_w, axis=1).astype(np.float32)

def rmse_local(a, b, axis=-1):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return np.sqrt(np.mean((a - b) ** 2, axis=axis) + 1e-12)

def prd_local(a, b, axis=-1):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    num = np.sqrt(np.sum((a - b) ** 2, axis=axis))
    den = np.sqrt(np.sum(a ** 2, axis=axis) + 1e-12)
    return 100.0 * (num / (den + 1e-12))

def cosine_sim_local(a, b, axis=-1):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    num = np.sum(a*b, axis=axis)
    den = (np.sqrt(np.sum(a*a, axis=axis) + 1e-12) * np.sqrt(np.sum(b*b, axis=axis) + 1e-12))
    return (num / (den + 1e-12)).astype(np.float32)

def eval_setting_on_simemg(model_name: str, weights_dir: Path, max_files=10, tail_frac=0.10, batch_size=256):
    weights_path = weights_dir / f"{model_name}_weights.best.weights.h5"
    if not weights_path.exists():
        raise FileNotFoundError(f"Missing weights: {weights_path}")
    model = load_model_weights(model_name, str(weights_path), signal_size=SAMPLES_LOCAL)

    paths = sorted(SIMEMG_DIR_LOCAL.glob('*.mat'))[:int(max_files)]
    all_in_snr = []
    all_out_snr = []
    all_file_rows = []
    for p in paths:
        two = load_simemg_two_channel_local(p)
        clean_1d, noisy_1d, info = infer_clean_noisy_local(two)
        clean_w = window_signal_1d_local(clean_1d, win=SAMPLES_LOCAL, hop=SAMPLES_LOCAL)
        noisy_w = window_signal_1d_local(noisy_1d, win=SAMPLES_LOCAL, hop=SAMPLES_LOCAL)
        if clean_w.shape[0] == 0:
            continue
        den_w = model_denoise_windows(model, noisy_w, batch_size=batch_size)
        snr_in = snr_db_windows_local(clean_w, noisy_w)
        snr_out = snr_db_windows_local(clean_w, den_w)
        k = max(1, int(np.ceil(tail_frac * len(snr_in))))
        worst_idx = np.argsort(snr_in)[:k]

        all_in_snr.append(snr_in)
        all_out_snr.append(snr_out)
        all_file_rows.append({
            'setting': weights_dir.name,
            'model': model_name,
            'path': str(p),
            'n_windows': int(len(snr_in)),
            'snr_in_mean': float(np.mean(snr_in)),
            'snr_out_mean': float(np.mean(snr_out)),
            'delta_snr_mean': float(np.mean(snr_out - snr_in)),
            'snr_in_tail_mean': float(np.mean(snr_in[worst_idx])),
            'snr_out_tail_mean': float(np.mean(snr_out[worst_idx])),
            'delta_snr_tail_mean': float(np.mean((snr_out - snr_in)[worst_idx])),
            'rmse_mean': float(np.mean(rmse_local(clean_w, den_w, axis=1))),
            'prd_mean': float(np.mean(prd_local(clean_w, den_w, axis=1))),
            'cos_mean': float(np.mean(cosine_sim_local(clean_w, den_w, axis=1))),
            'noisy_idx': info['noisy_idx'],
        })

    if not all_in_snr:
        raise RuntimeError('No valid SimEMG windows found')
    in_snr = np.concatenate(all_in_snr)
    out_snr = np.concatenate(all_out_snr)
    delta = out_snr - in_snr
    k = max(1, int(np.ceil(tail_frac * len(in_snr))))
    worst_idx = np.argsort(in_snr)[:k]
    summary = {
        'setting': weights_dir.name,
        'model': model_name,
        'n_files': int(len(all_file_rows)),
        'n_windows': int(len(in_snr)),
        'snr_in_mean': float(np.mean(in_snr)),
        'snr_out_mean': float(np.mean(out_snr)),
        'delta_snr_mean': float(np.mean(delta)),
        'snr_in_tail_mean': float(np.mean(in_snr[worst_idx])),
        'snr_out_tail_mean': float(np.mean(out_snr[worst_idx])),
        'delta_snr_tail_mean': float(np.mean(delta[worst_idx])),
    }
    return summary, pd.DataFrame(all_file_rows)

# Compare CENSD-trained vs fixed-SNR-trained settings using ONE architecture (Dual_FreqDAE)
settings = [Path('0221_FIXED'), Path('0221_snr0db'), Path('0221_snr-3db'), Path('0221_snr3db')]
model_name = 'Dual_FreqDAE'
summaries = []
file_tables = []
print('[INFO] SIMEMG_DIR_LOCAL:', SIMEMG_DIR_LOCAL, 'exists=', SIMEMG_DIR_LOCAL.exists())
for sd in settings:
    try:
        s, df_files = eval_setting_on_simemg(model_name, sd, max_files=10, tail_frac=0.10, batch_size=256)
        summaries.append(s)
        file_tables.append(df_files)
        print('[OK]', s)
    except Exception as e:
        print('[ERR]', sd, repr(e))

df_cmp = pd.DataFrame(summaries)
print('[INFO] Comparison: higher delta_snr_tail_mean is better (tail robustness)')
display(df_cmp.sort_values('delta_snr_tail_mean', ascending=False))
if file_tables:
    df_files_all = pd.concat(file_tables, ignore_index=True)
    display(df_files_all.sort_values('delta_snr_tail_mean', ascending=False).head(20))

[INFO] SIMEMG_DIR_LOCAL: data/SimEMG exists= True


/home/dhc99/anaconda3/envs/ECGDENOISE/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
